# LoRA Fine-tune LLaVA-1.5-7B — Endoscopy AI

**Platform:** Kaggle (2x T4 = 32GB VRAM)  
**Model:** llava-hf/llava-1.5-7b-hf  
**Method:** LoRA via HuggingFace PEFT  
**Dataset:** HyperKvasir + GPT-4o captions (metadata.jsonl)

## Setup trước khi chạy
1. Upload `data/llava_dataset/` lên Kaggle Dataset
2. Add dataset vào notebook này
3. Bật GPU T4 x2 trong Settings → Accelerator
4. Chạy từng cell theo thứ tự

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers==4.40.0 peft==0.10.0 bitsandbytes==0.43.1 \
                accelerate==0.29.3 datasets==2.18.0 Pillow tqdm

In [ ]:
# ── Cell 2: Imports & config ──────────────────────────────────────────────────
import os, json
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    LlavaForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType

# ── Paths (adjust if dataset mount point differs) ─────────────────────────────
DATASET_DIR  = Path("/kaggle/input/endoscopy-llava-dataset")   # your Kaggle dataset name
OUTPUT_DIR   = Path("/kaggle/working/llava-lora-endoscopy")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID     = "llava-hf/llava-1.5-7b-hf"
MAX_LENGTH   = 512
BATCH_SIZE   = 2       # 2x T4: keep low to avoid OOM
GRAD_ACCUM   = 8       # effective batch = 16
EPOCHS       = 3
LR           = 2e-4

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Cell 3: Load model in 4-bit quantization (saves VRAM) ────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "right"

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False
print("Model loaded")

In [ ]:
# ── Cell 4: Apply LoRA ────────────────────────────────────────────────────────
# Chỉ train attention layers của language model, bỏ qua vision encoder
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,               # rank — tăng lên 32 nếu muốn chất lượng cao hơn
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[    # chỉ train language model layers
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.1% trainable params (~8M / 7B)

In [ ]:
# ── Cell 5: Dataset ───────────────────────────────────────────────────────────
class EndoscopyDataset(Dataset):
    def __init__(self, metadata_path: Path, dataset_dir: Path, processor, max_length: int):
        self.dataset_dir = dataset_dir
        self.processor = processor
        self.max_length = max_length
        self.samples = []

        with open(metadata_path) as f:
            for line in f:
                entry = json.loads(line.strip())
                img_path = dataset_dir / entry["image"]
                if img_path.exists():
                    convs = entry["conversations"]
                    question = convs[0]["value"]  # human turn
                    answer   = convs[1]["value"]  # gpt turn
                    self.samples.append((img_path, question, answer))

        print(f"Dataset: {len(self.samples)} samples loaded")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, question, answer = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        # LLaVA conversation format
        prompt = f"USER: {question}\nASSISTANT: {answer}"

        inputs = self.processor(
            text=prompt,
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
        )
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["labels"] = inputs["input_ids"].clone()
        return inputs


metadata_path = DATASET_DIR / "metadata.jsonl"
dataset = EndoscopyDataset(metadata_path, DATASET_DIR, processor, MAX_LENGTH)

# 90/10 train/val split
val_size  = max(1, int(len(dataset) * 0.1))
train_size = len(dataset) - val_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
# ── Cell 6: Training loop ─────────────────────────────────────────────────────
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))

best_val_loss = float("inf")

for epoch in range(EPOCHS):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")):
        batch = {k: v.cuda() for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()
        train_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    avg_train = train_loss / len(train_loader)

    # ── Validation ─────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            outputs = model(**batch)
            val_loss += outputs.loss.item()

    avg_val = val_loss / len(val_loader)
    print(f"Epoch {epoch+1}: train_loss={avg_train:.4f}  val_loss={avg_val:.4f}")

    # Save best checkpoint
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained(OUTPUT_DIR / "best")
        processor.save_pretrained(OUTPUT_DIR / "best")
        print(f"  ✔ Saved best model (val_loss={avg_val:.4f})")

print("Training complete!")

In [ ]:
# ── Cell 7: Quick inference test ──────────────────────────────────────────────
from peft import PeftModel

model.eval()

# Lấy 1 ảnh bất kỳ từ dataset để test
test_img_path, test_question, test_answer_gt = dataset.samples[0]
test_image = Image.open(test_img_path).convert("RGB")

prompt = f"USER: {test_question}\nASSISTANT:"
inputs = processor(text=prompt, images=test_image, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        temperature=1.0,
    )

generated = processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=== Ground truth ===")
print(test_answer_gt)
print("\n=== Model output ===")
print(generated)

In [ ]:
# ── Cell 8: Export LoRA weights (zip để download) ─────────────────────────────
import shutil

# Save final checkpoint
model.save_pretrained(OUTPUT_DIR / "final")
processor.save_pretrained(OUTPUT_DIR / "final")

# Zip để download về
shutil.make_archive("/kaggle/working/llava-lora-weights", "zip", OUTPUT_DIR / "best")
print("Done! Download llava-lora-weights.zip from output panel.")